# Heart Disease MLOps — Google Colab End-to-End Notebook

**Course:** AIMLCZG523 · Assignment 01

This notebook is the easiest entry point for running the data and model workflow in Google Colab. It downloads the processed Cleveland Heart Disease dataset, performs EDA, compares Logistic Regression and Random Forest using stratified cross-validation, tracks experiments with MLflow, evaluates on a held-out test set, and exports a reusable preprocessing + model pipeline.

**Important:** This is an educational classification project, not a medical device. Its outputs must not be used for diagnosis or treatment.

### How to run
1. Upload this `.ipynb` file to Google Colab.
2. Choose **Runtime → Run all**. The install cell may take a minute.
3. Let every cell finish. The final cell creates a ZIP of model and evidence artifacts for download.
4. Use the exported model and real metrics in the report. Docker, GitHub Actions, and Kubernetes still need their own execution/verification; a Colab notebook alone does not prove those deployment tasks.

In [ ]:
# Cell 1 — Install the packages used by this notebook.
%pip -q install pandas numpy scikit-learn matplotlib seaborn mlflow joblib

In [ ]:
# Cell 2 — Imports, reproducibility, and workspace.
import io
import json
import os
import shutil
import zipfile
import warnings
from pathlib import Path
from urllib.request import urlopen

import joblib
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay, classification_report
)
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from IPython.display import display

SEED = 42
ROOT = Path("/content/heart_disease_mlops")
DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "models"
ARTIFACT_DIR = ROOT / "artifacts"
EDA_DIR = ARTIFACT_DIR / "eda"
for folder in [DATA_DIR, MODEL_DIR, EDA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal"
]
COLUMNS = FEATURES + ["target"]
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

print("Workspace:", ROOT)
print("Random seed:", SEED)

## 1. Acquire and validate the data

The original target is encoded from 0 to 4. For this binary assignment, target `0` is mapped to class `0`, while values greater than `0` are mapped to class `1`. The `?` values in the source are treated as missing. Predictor imputation is fitted inside the model pipeline so validation/test information is not used to fit preprocessing.

In [ ]:
# Cell 3 — Download the public processed Cleveland file.
# If the source is temporarily unavailable, upload processed.cleveland.data when prompted.
try:
    with urlopen(DATA_URL, timeout=30) as response:
        raw_text = response.read().decode("utf-8")
    print("Downloaded dataset from the UCI repository.")
except Exception as exc:
    print("Automatic download failed:", repr(exc))
    try:
        from google.colab import files
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        raw_text = uploaded[uploaded_name].decode("utf-8")
        print("Loaded uploaded file:", uploaded_name)
    except Exception as upload_exc:
        raise RuntimeError("Could not download data. Upload processed.cleveland.data and rerun this cell.") from upload_exc

df = pd.read_csv(
    io.StringIO(raw_text), header=None, names=COLUMNS,
    na_values="?", skipinitialspace=True
)
for column in COLUMNS:
    df[column] = pd.to_numeric(df[column], errors="coerce")
df = df.dropna(subset=["target"]).copy()
df["target"] = (df["target"] > 0).astype(int)
if df.empty or not set(df["target"].unique()).issubset({0, 1}):
    raise ValueError("Dataset is empty or target normalization failed.")

df.to_csv(DATA_DIR / "heart.csv", index=False)
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("Target counts:")
display(df["target"].value_counts().sort_index().rename(index={0:"No disease (0)", 1:"Disease present (1)"}).to_frame("count"))
print("Missing values by column:")
display(df.isna().sum().to_frame("missing_values"))
display(df.head())

## 2. Exploratory data analysis

The following plots show class balance, predictor distributions, and correlations. Correlation is descriptive and does not establish causation. Review the plots and write your own short observations in the report after running the notebook.

In [ ]:
# Cell 4 — Class balance and feature distributions.
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x="target", ax=ax)
ax.set(title="Target class balance", xlabel="Target (0 = no disease, 1 = disease present)", ylabel="Records")
fig.tight_layout()
fig.savefig(EDA_DIR / "class_balance.png", dpi=160, bbox_inches="tight")
plt.show()

df[FEATURES].hist(figsize=(15, 11), bins=20, edgecolor="white")
plt.suptitle("Feature distributions", y=1.01)
plt.tight_layout()
plt.savefig(EDA_DIR / "feature_distributions.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(df.corr(numeric_only=True), cmap="vlag", center=0, annot=True, fmt=".2f", ax=ax)
ax.set_title("Correlation heatmap")
fig.tight_layout()
fig.savefig(EDA_DIR / "correlation_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["age", "thalach", "oldpeak"]):
    sns.boxplot(data=df, x="target", y=col, ax=ax)
    ax.set_title(f"{col} by target")
fig.tight_layout()
fig.savefig(EDA_DIR / "selected_features_by_target.png", dpi=160, bbox_inches="tight")
plt.show()

print("Class proportions:")
display(df["target"].value_counts(normalize=True).sort_index().rename("proportion").to_frame())

## 3. Feature engineering and model selection

We use median imputation and standard scaling in a scikit-learn pipeline. The data is split once into stratified training and test partitions. Hyperparameter search and cross-validation use the training split only; the held-out test set is reserved for final evaluation. Both models use ROC-AUC as the tuning objective and report accuracy, precision, recall, and ROC-AUC.

In [ ]:
# Cell 5 — Build leakage-safe pipelines and split the data.
X = df[FEATURES].copy()
y = df["target"].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def make_pipeline(classifier):
    preprocessing = ColumnTransformer(
        transformers=[(
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            FEATURES
        )],
        remainder="drop"
    )
    return Pipeline([("preprocess", preprocessing), ("classifier", classifier)])

candidates = {
    "logistic_regression": (
        make_pipeline(LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)),
        {"classifier__C": [0.1, 1.0, 10.0]}
    ),
    "random_forest": (
        make_pipeline(RandomForestClassifier(class_weight="balanced", random_state=SEED)),
        {"classifier__n_estimators": [150, 300], "classifier__max_depth": [None, 5, 10]}
    )
}
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Training target balance:")
display(y_train.value_counts(normalize=True).sort_index().rename("proportion").to_frame())

## 4. MLflow experiment tracking and evaluation

Runs are stored locally in this Colab runtime. They remain available while the runtime persists; download the exported evidence ZIP at the end if you need a copy. Each model run logs hyperparameters, cross-validation metrics, held-out metrics, plots, a classification report, and the fitted pipeline.

In [ ]:
# Cell 6 — Tune, evaluate, and log both models.
mlflow.set_tracking_uri(f"file:{ROOT / 'mlruns'}")
mlflow.set_experiment("heart-disease-classification-colab")

SCORING = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc"
}
results = []
run_summaries = []

def evaluate_model(y_true, pred, proba):
    return {
        "test_accuracy": float(accuracy_score(y_true, pred)),
        "test_precision": float(precision_score(y_true, pred, zero_division=0)),
        "test_recall": float(recall_score(y_true, pred, zero_division=0)),
        "test_roc_auc": float(roc_auc_score(y_true, proba))
    }

for model_name, (pipeline, param_grid) in candidates.items():
    with mlflow.start_run(run_name=model_name):
        search = GridSearchCV(
            estimator=pipeline, param_grid=param_grid, scoring="roc_auc",
            cv=cv, n_jobs=-1, refit=True, return_train_score=False
        )
        search.fit(X_train, y_train)
        best_pipeline = search.best_estimator_

        cv_scores = cross_validate(
            best_pipeline, X_train, y_train, cv=cv, scoring=SCORING,
            n_jobs=-1, return_train_score=False
        )
        predictions = best_pipeline.predict(X_test)
        probabilities = best_pipeline.predict_proba(X_test)[:, 1]
        test_metrics = evaluate_model(y_test, predictions, probabilities)

        mlflow.log_param("model", model_name)
        mlflow.log_param("random_seed", SEED)
        mlflow.log_param("cv_folds", 5)
        mlflow.log_params({f"best_{key}": str(value) for key, value in search.best_params_.items()})
        mlflow.log_metric("best_cv_roc_auc", float(search.best_score_))
        for metric_name, values in cv_scores.items():
            mlflow.log_metric(f"cv_{metric_name}_mean", float(np.mean(values)))
            mlflow.log_metric(f"cv_{metric_name}_std", float(np.std(values)))
        mlflow.log_metrics(test_metrics)

        model_artifact_dir = ARTIFACT_DIR / model_name
        model_artifact_dir.mkdir(parents=True, exist_ok=True)

        fig, ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay.from_predictions(y_test, predictions, ax=ax)
        fig.tight_layout()
        cm_path = model_artifact_dir / "confusion_matrix.png"
        fig.savefig(cm_path, dpi=160, bbox_inches="tight")
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(5, 4))
        RocCurveDisplay.from_predictions(y_test, probabilities, ax=ax)
        fig.tight_layout()
        roc_path = model_artifact_dir / "roc_curve.png"
        fig.savefig(roc_path, dpi=160, bbox_inches="tight")
        plt.close(fig)

        report_dict = classification_report(
            y_test, predictions, output_dict=True, zero_division=0
        )
        report_path = model_artifact_dir / "classification_report.json"
        report_path.write_text(json.dumps(report_dict, indent=2))
        mlflow.log_artifact(str(cm_path))
        mlflow.log_artifact(str(roc_path))
        mlflow.log_artifact(str(report_path))
        # MLflow 2.x/3.x compatible artifact_path form.
        mlflow.sklearn.log_model(best_pipeline, artifact_path="model")

        results.append({
            "name": model_name,
            "pipeline": best_pipeline,
            "cv_auc": float(search.best_score_),
            "params": search.best_params_,
            "metrics": test_metrics
        })
        run_summaries.append({
            "model": model_name,
            "best_cv_roc_auc": float(search.best_score_),
            **test_metrics,
            "best_params": search.best_params_
        })
        print(f"Finished: {model_name}")

comparison = pd.DataFrame([
    {"model": r["name"], "mean_cv_roc_auc": r["cv_auc"], **r["metrics"], "best_params": str(r["params"])}
    for r in results
]).sort_values("mean_cv_roc_auc", ascending=False).reset_index(drop=True)
display(comparison)
comparison.to_csv(ARTIFACT_DIR / "model_comparison.csv", index=False)
(ARTIFACT_DIR / "mlflow_run_summary.json").write_text(json.dumps(run_summaries, indent=2))

## 5. Select and export the final pipeline

The winning model is selected by mean cross-validated ROC-AUC on the training partition. The held-out test metrics are reported separately and are not used to select the winner. The saved artifact includes preprocessing and classification together, so inference uses the same transformations as training.

In [ ]:
# Cell 7 — Save model, metrics, and a sample inference.
winner = max(results, key=lambda item: item["cv_auc"])
model_path = MODEL_DIR / "heart_pipeline.joblib"
summary_path = MODEL_DIR / "heart_pipeline.json"

joblib.dump(winner["pipeline"], model_path)
summary = {
    "selected_model": winner["name"],
    "selection_basis": "highest mean cross-validated ROC-AUC on training split",
    "best_params": winner["params"],
    "test_metrics": winner["metrics"],
    "seed": SEED,
    "feature_order": FEATURES,
    "dataset": "UCI Heart Disease processed Cleveland subset",
    "note": "Educational example only; not a medical device."
}
summary_path.write_text(json.dumps(summary, indent=2))

sample = X_test.iloc[[0]]
sample_prediction = int(winner["pipeline"].predict(sample)[0])
sample_probabilities = winner["pipeline"].predict_proba(sample)[0]
print("Selected model:", winner["name"])
print("Saved model:", model_path)
print("Saved summary:", summary_path)
print("Sample prediction:", sample_prediction)
print("Sample positive-class probability:", round(float(sample_probabilities[1]), 4))
display(pd.DataFrame([summary["test_metrics"]]))

## 6. Run a simple inference check

This is a local notebook inference check, not a deployed API. The API, Docker image, CI pipeline, and Kubernetes configuration are in the accompanying project bundle and must be tested separately.

In [ ]:
# Cell 8 — Reload the serialized pipeline and verify it can predict.
reloaded_model = joblib.load(model_path)
probe = X_test.iloc[[0]]
assert len(reloaded_model.predict(probe)) == 1
assert reloaded_model.predict_proba(probe).shape == (1, 2)
print("Serialization smoke test passed.")
print("Prediction:", int(reloaded_model.predict(probe)[0]))
print("Probabilities:", reloaded_model.predict_proba(probe)[0].round(4))

## 7. Package the evidence for download

The final ZIP includes the dataset copy, exported model, metrics, EDA plots, model comparison plots, and local MLflow tracking files. Download it before closing the Colab runtime. Do not include patient-specific real data; this assignment uses a public educational dataset.

In [ ]:
# Cell 9 — Create and download an artifact bundle.
bundle_path = Path("/content/heart_disease_mlops_colab_artifacts.zip")
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [DATA_DIR, MODEL_DIR, ARTIFACT_DIR, ROOT / "mlruns"]:
        if folder.exists():
            for file_path in folder.rglob("*"):
                if file_path.is_file():
                    zf.write(file_path, file_path.relative_to(ROOT))

print("Created:", bundle_path, "| size:", round(bundle_path.stat().st_size / 1024, 1), "KB")
try:
    from google.colab import files
    files.download(str(bundle_path))
except Exception:
    print("Not running in Colab. Download the file from the notebook runtime file browser.")

## 8. Remaining assignment work outside this notebook

Use the metrics and artifacts from this run in your report. Then, from the project folder:

1. Install `requirements.txt` in a clean environment and run `pytest -q`.
2. Build and run the API container with Docker; test `/health` and `/predict`.
3. Push the code to GitHub and verify the Actions workflow succeeds.
4. Deploy to Minikube/Docker Desktop Kubernetes or a cloud cluster and capture actual status/endpoint screenshots.
5. Start Prometheus/Grafana and capture a dashboard showing request counts, latency, and errors.
6. Record the required short video showing the workflow end-to-end.

Do not claim that a workflow, container, or deployment succeeded unless you actually ran it and captured the evidence.